
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>




# SQL UDFs and Control Flow


## Learning Objectives
By the end of this lesson, you should be able to:
* Define and registering SQL UDFs
* Describe the security model used for sharing SQL UDFs
* Use **`CASE`** / **`WHEN`** statements in SQL code
* Leverage **`CASE`** / **`WHEN`** statements in SQL UDFs for custom control flow



## Run Setup
Run the following cell to setup your environment.

In [0]:
%run ./Includes/Classroom-Setup-02.7A

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


Resetting the learning environment:
| removing the working directory "dbfs:/mnt/dbacademy-users/shifajamali55@gmail.com/data-engineering-with-databricks"...(0 seconds)

Skipping install of existing datasets to "dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04"

Validating the locally installed datasets:
| listing local files...(8 seconds)
| validation completed...(8 seconds total)

Creating & using the schema "shifajamali55_soeb_da_dewd" in the catalog "hive_metastore"...(6 seconds)

Cloning the "sales" table from "dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/ecommerce/delta/sales_hist"....(29 seconds)
Cloning the "events" table from "dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/ecommerce/delta/events_hist"....(9 seconds)
Cloning the "events_raw" table from "dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v04/ecommerce/delta/events_raw"....(8 seconds)
Cloning the "item_lookup" table from "dbfs:/mnt/dbacademy-datasets/data-engineer-le


## User-Defined Functions

User Defined Functions (UDFs) in Spark SQL allow you to register custom SQL logic as functions in a database, making these methods reusable anywhere SQL can be run on Databricks. These functions are registered natively in SQL and maintain all of the optimizations of Spark when applying custom logic to large datasets.

At minimum, creating a SQL UDF requires a function name, optional parameters, the type to be returned, and some custom logic.

Below, a simple function named **`sale_announcement`** takes an **`item_name`** and **`item_price`** as parameters. It returns a string that announces a sale for an item at 80% of its original price.

In [0]:
CREATE OR REPLACE FUNCTION sale_announcement(item_name STRING, item_price INT)
RETURNS STRING
RETURN concat("The ", item_name, " is on sale for $", round(item_price * 0.8, 0));

SELECT *, sale_announcement(name, price) AS message FROM item_lookup

item_id,name,price,message
M_PREM_Q,Premium Queen Mattress,1795.0,The Premium Queen Mattress is on sale for $1436
M_STAN_F,Standard Full Mattress,945.0,The Standard Full Mattress is on sale for $756
M_PREM_F,Premium Full Mattress,1695.0,The Premium Full Mattress is on sale for $1356
M_PREM_T,Premium Twin Mattress,1095.0,The Premium Twin Mattress is on sale for $876
M_PREM_K,Premium King Mattress,1995.0,The Premium King Mattress is on sale for $1596
P_DOWN_S,Standard Down Pillow,119.0,The Standard Down Pillow is on sale for $95
M_STAN_Q,Standard Queen Mattress,1045.0,The Standard Queen Mattress is on sale for $836
M_STAN_K,Standard King Mattress,1195.0,The Standard King Mattress is on sale for $956
M_STAN_T,Standard Twin Mattress,595.0,The Standard Twin Mattress is on sale for $476
P_FOAM_S,Standard Foam Pillow,59.0,The Standard Foam Pillow is on sale for $47



Note that this function is applied to all values of the column in a parallel fashion within the Spark processing engine. SQL UDFs are an efficient way to define custom logic that is optimized for execution on Databricks.


## Scoping and Permissions of SQL UDFs
SQL user-defined functions:
- Persist between execution environments (which can include notebooks, DBSQL queries, and jobs).
- Exist as objects in the metastore and are governed by the same Table ACLs as databases, tables, or views.
- To **create** a SQL UDF, you need **`USE CATALOG`** on the catalog, and **`USE SCHEMA`** and **`CREATE FUNCTION`** on the schema.
- To **use** a SQL UDF, you need **`USE CATALOG`** on the catalog, **`USE SCHEMA`** on the schema, and **`EXECUTE`** on the function.

We can use **`DESCRIBE FUNCTION`** to see where a function was registered and basic information about expected inputs and what is returned (and even more information with **`DESCRIBE FUNCTION EXTENDED`**).

In [0]:
DESCRIBE FUNCTION EXTENDED sale_announcement

function_desc
Function: hive_metastore.shifajamali55_soeb_da_dewd.sale_announcement
Type: SCALAR
Input: item_name STRING
item_price INT
Returns: STRING
Deterministic: true
Data Access: CONTAINS SQL
Configs: spark.sql.hive.convertCTAS=true
spark.sql.legacy.createHiveTableByDefault=false
spark.sql.parquet.compression.codec=snappy



Note that the **`Body`** field at the bottom of the function description shows the SQL logic used in the function itself.



## Simple Control Flow Functions

Combining SQL UDFs with control flow in the form of **`CASE`** / **`WHEN`** clauses provides optimized execution for control flows within SQL workloads. The standard SQL syntactic construct **`CASE`** / **`WHEN`** allows the evaluation of multiple conditional statements with alternative outcomes based on table contents.

Here, we demonstrate wrapping this control flow logic in a function that will be reusable anywhere we can execute SQL.

In [0]:
CREATE OR REPLACE FUNCTION item_preference(name STRING, price INT)
RETURNS STRING
RETURN CASE 
  WHEN name = "Standard Queen Mattress" THEN "This is my default mattress"
  WHEN name = "Premium Queen Mattress" THEN "This is my favorite mattress"
  WHEN price > 100 THEN concat("I'd wait until the ", name, " is on sale for $", round(price * 0.8, 0))
  ELSE concat("I don't need a ", name)
END;

SELECT *, item_preference(name, price) FROM item_lookup

item_id,name,price,"hive_metastore.shifajamali55_soeb_da_dewd.item_preference(name, price)"
M_PREM_Q,Premium Queen Mattress,1795.0,This is my favorite mattress
M_STAN_F,Standard Full Mattress,945.0,I'd wait until the Standard Full Mattress is on sale for $756
M_PREM_F,Premium Full Mattress,1695.0,I'd wait until the Premium Full Mattress is on sale for $1356
M_PREM_T,Premium Twin Mattress,1095.0,I'd wait until the Premium Twin Mattress is on sale for $876
M_PREM_K,Premium King Mattress,1995.0,I'd wait until the Premium King Mattress is on sale for $1596
P_DOWN_S,Standard Down Pillow,119.0,I'd wait until the Standard Down Pillow is on sale for $95
M_STAN_Q,Standard Queen Mattress,1045.0,This is my default mattress
M_STAN_K,Standard King Mattress,1195.0,I'd wait until the Standard King Mattress is on sale for $956
M_STAN_T,Standard Twin Mattress,595.0,I'd wait until the Standard Twin Mattress is on sale for $476
P_FOAM_S,Standard Foam Pillow,59.0,I don't need a Standard Foam Pillow




While the examples provided here are simple, these same basic principles can be used to add custom computations and logic for native execution in Spark SQL. 

Especially for enterprises that might be migrating users from systems with many defined procedures or custom-defined formulas, SQL UDFs can allow a handful of users to define the complex logic needed for common reporting and analytic queries.


 
Run the following cell to delete the tables and files associated with this lesson.

In [0]:
%python
DA.cleanup()

Resetting the learning environment:
| dropping the schema "shifajamali55_soeb_da_dewd"...(3 seconds)
| removing the working directory "dbfs:/mnt/dbacademy-users/shifajamali55@gmail.com/data-engineering-with-databricks"...(0 seconds)

Validating the locally installed datasets:
| listing local files...(7 seconds)
| validation completed...(7 seconds total)



&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>